# Chapter 21 — Treatment Effects
## University of Neuchâtel — Python for Econometrics

---

## What you will learn

| Section | Topic |
|---|---|
| 1 | Potential outcomes framework |
| 2 | ATE, ATT, ATU — what are we estimating? |
| 3 | Selection bias |
| 4 | RCT — the gold standard |
| 5 | Differences-in-Differences (DiD) |
| 6 | Regression Discontinuity Design (RDD) |
| 7 | Propensity Score Matching (PSM) |
| 8 | Local Average Treatment Effect (LATE) from IV |

**Format:** One cell per step. Manual calculation first, statsmodels second, `np.isclose()` verification.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os
try:
    from linearmodels.iv import IV2SLS
    HAS_LM = True
except ImportError:
    HAS_LM = False

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

---
## 1 — The Potential Outcomes Framework (Rubin Causal Model)

### 1.1 Notation

For each individual $i$, define:

- $Y_i(1)$: **potential outcome if treated** ($D_i = 1$)
- $Y_i(0)$: **potential outcome if not treated** ($D_i = 0$)
- $D_i \in \{0, 1\}$: treatment indicator

The **individual treatment effect** is:

$$\tau_i = Y_i(1) - Y_i(0)$$

### 1.2 The Fundamental Problem of Causal Inference

We **never observe both** $Y_i(1)$ and $Y_i(0)$ for the same person at the same time.
We observe only:

$$Y_i = D_i \cdot Y_i(1) + (1-D_i) \cdot Y_i(0)$$

The counterfactual — what would have happened under the opposite treatment — is missing.

### 1.3 Average Treatment Effects

Because $\tau_i$ is unobservable, we estimate **averages**:

| Estimand | Formula | Meaning |
|---|---|---|
| **ATE** | $E[Y_i(1) - Y_i(0)]$ | Average over the **entire population** |
| **ATT** | $E[Y_i(1) - Y_i(0) \mid D_i=1]$ | Average over the **treated** |
| **ATU** | $E[Y_i(1) - Y_i(0) \mid D_i=0]$ | Average over the **untreated** |
| **LATE** | $E[Y_i(1) - Y_i(0) \mid \text{complier}]$ | Average over IV **compliers** |

**ATE = ATT = ATU** only when treatment is randomly assigned (no selection bias).

### Step 1.1 — Simulate potential outcomes

In [ ]:
np.random.seed(42)
n = 1000
# Heterogeneous individual treatment effects
ability = np.random.normal(100, 15, n)     # individual ability
Y0 = 20 + 0.3*ability + np.random.normal(0, 5, n)   # earnings without training
tau_i = 5 + 0.05*(ability - 100) + np.random.normal(0, 2, n)  # individual TE (higher for able)
Y1 = Y0 + tau_i                                                # earnings with training
print('Potential outcomes (first 5 individuals):')
print(pd.DataFrame({'Y0': Y0[:5].round(2), 'Y1': Y1[:5].round(2),
                    'tau_i': tau_i[:5].round(2), 'ability': ability[:5].round(1)}))

In [ ]:
ATE_true = tau_i.mean()
print(f'True ATE = E[Y(1)-Y(0)] = {ATE_true:.3f}')
print(f'  (average over all {n} individuals)')

---
## 2 — Selection Bias

### 2.1 The Problem

When treatment is **not randomly assigned**, individuals self-select based on
characteristics that also affect outcomes.

**Naive estimator** (simple difference in means):

$$\underbrace{E[Y_i \mid D_i=1] - E[Y_i \mid D_i=0]}_{\text{observed difference}} = \underbrace{E[Y_i(1)-Y_i(0) \mid D_i=1]}_{ATT} + \underbrace{E[Y_i(0) \mid D_i=1] - E[Y_i(0) \mid D_i=0]}_{\text{selection bias}}$$

The naive estimate = ATT + **selection bias**.

Selection bias arises because treated individuals would have done better **even without treatment**
(e.g., more able people choose to take training AND earn more regardless).

### Step 2.1 — Create selection: able individuals self-select into treatment

In [ ]:
# High-ability people are more likely to take training
prob_treat = 1 / (1 + np.exp(-(ability - 100)/10))  # logistic
D = np.random.binomial(1, prob_treat)
Y_obs = D * Y1 + (1-D) * Y0
df_po = pd.DataFrame({'Y_obs': Y_obs, 'D': D, 'Y0': Y0, 'Y1': Y1,
                      'tau_i': tau_i, 'ability': ability})
print(f'Treated:   {D.sum()} individuals, mean ability = {ability[D==1].mean():.1f}')
print(f'Untreated: {(1-D).sum()} individuals, mean ability = {ability[D==0].mean():.1f}')

In [ ]:
naive_diff = df_po.loc[df_po['D']==1,'Y_obs'].mean() - df_po.loc[df_po['D']==0,'Y_obs'].mean()
ATT_true   = df_po.loc[df_po['D']==1,'tau_i'].mean()
selection_bias = naive_diff - ATT_true
print(f'Naive difference in means: {naive_diff:.3f}')
print(f'True ATT:                  {ATT_true:.3f}')
print(f'Selection bias:            {selection_bias:.3f}')
print(f'Check: ATT + bias = {ATT_true:.3f} + {selection_bias:.3f} = {ATT_true+selection_bias:.3f} ≈ {naive_diff:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(df_po.loc[df_po['D']==1,'ability'], bins=25, alpha=0.7, label='Treated', density=True)
axes[0].hist(df_po.loc[df_po['D']==0,'ability'], bins=25, alpha=0.7, label='Untreated', density=True)
axes[0].set_xlabel('Ability')
axes[0].set_title('Ability Distribution by Treatment Status')
axes[0].legend()
axes[1].scatter(df_po.loc[df_po['D']==0,'ability'], df_po.loc[df_po['D']==0,'Y0'],
                alpha=0.3, s=10, label='Untreated (Y0)')
axes[1].scatter(df_po.loc[df_po['D']==1,'ability'], df_po.loc[df_po['D']==1,'Y1'],
                alpha=0.3, s=10, color='red', label='Treated (Y1)')
axes[1].set_xlabel('Ability')
axes[1].set_ylabel('Earnings')
axes[1].set_title('Selection: able people self-select into treatment')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch21_01_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — Randomised Control Trial (RCT): The Gold Standard

### 3.1 Why Randomisation Solves Selection Bias

With **random assignment**, $D_i$ is independent of potential outcomes:

$$D_i \perp (Y_i(0), Y_i(1))$$

This means:

$$E[Y_i(0) \mid D_i=1] = E[Y_i(0) \mid D_i=0] = E[Y_i(0)]$$

So selection bias = 0, and:

$$E[Y_i \mid D_i=1] - E[Y_i \mid D_i=0] = ATE = ATT$$

The simple difference in means **identifies the ATE** under random assignment.

### 3.2 OLS with Random Treatment Dummy

$$Y_i = \alpha + \tau D_i + \varepsilon_i$$

The OLS estimator of $\tau$ equals the difference in means:

$$\hat{\tau} = \bar{Y}_1 - \bar{Y}_0$$

### Step 3.1 — Simulate RCT (flip a coin for treatment)

In [ ]:
np.random.seed(10)
D_rct = np.random.binomial(1, 0.5, n)
Y_rct = D_rct * Y1 + (1-D_rct) * Y0
df_rct = pd.DataFrame({'Y': Y_rct, 'D': D_rct, 'ability': ability})
print(f'RCT — Treated: {D_rct.sum()}, Untreated: {(1-D_rct).sum()}')
print(f'Mean ability treated:   {ability[D_rct==1].mean():.2f}')
print(f'Mean ability untreated: {ability[D_rct==0].mean():.2f}')
print('(Should be similar — random assignment balances covariates)')

### Step 3.2 — Simple difference in means (manual ATE estimate)

In [ ]:
Y1_bar = df_rct.loc[df_rct['D']==1,'Y'].mean()
Y0_bar = df_rct.loc[df_rct['D']==0,'Y'].mean()
ATE_rct_manual = Y1_bar - Y0_bar
print(f'Mean Y (treated):   {Y1_bar:.3f}')
print(f'Mean Y (untreated): {Y0_bar:.3f}')
print(f'ATE estimate (manual diff-in-means): {ATE_rct_manual:.3f}')
print(f'True ATE:                            {ATE_true:.3f}')

### Step 3.3 — OLS regression (should give the same estimate)

In [ ]:
res_rct = smf.ols('Y ~ D', data=df_rct).fit()
print(f'OLS treatment effect: {res_rct.params["D"]:.3f}')
print(f'True ATE:             {ATE_true:.3f}')
print(f'Match (manual vs OLS): {np.isclose(ATE_rct_manual, res_rct.params["D"])}')

In [ ]:
# OLS with controls (adds precision, does not change point estimate under RCT)
res_rct_ctrl = smf.ols('Y ~ D + ability', data=df_rct).fit()
print(f'OLS + controls: {res_rct_ctrl.params["D"]:.3f}')
print(f'OLS no controls: {res_rct.params["D"]:.3f}')
print(f'SE (no controls): {res_rct.bse["D"]:.3f}')
print(f'SE (+ controls):  {res_rct_ctrl.bse["D"]:.3f}')
print('Controls improve precision (lower SE) but do not change the point estimate under RCT')

---
## 4 — Differences-in-Differences (DiD)

### 4.1 Setup

When treatment is non-random but we have **pre- and post-treatment data**,
DiD eliminates time-invariant selection bias.

**Model:**
$$Y_{it} = \alpha + \beta_1 D_i + \beta_2 Post_t + \delta (D_i \times Post_t) + \varepsilon_{it}$$

$\delta$ = DiD treatment effect

**Key assumption (parallel trends):** In the absence of treatment, treated and control groups
would have followed the **same time trend**.

$$E[Y_i(0)_{post} - Y_i(0)_{pre} \mid D_i=1] = E[Y_i(0)_{post} - Y_i(0)_{pre} \mid D_i=0]$$

### 4.2 DiD = ATT under parallel trends

Under parallel trends, the DiD estimator identifies the **Average Treatment Effect on the Treated (ATT)**.

### Step 4.1 — Load real DiD data: kielmc

In [ ]:
kielmc = woo.dataWoo('kielmc')
kielmc['y81'] = (kielmc['year'] == 1981).astype(int)
kielmc['nearinc_y81'] = kielmc['nearinc'] * kielmc['y81']
print('kielmc shape:', kielmc.shape)
print(kielmc[['rprice','year','nearinc','y81']].head(8))

### Step 4.2 — DiD via group means (manual)

In [ ]:
m_t1 = kielmc.loc[(kielmc['nearinc']==1)&(kielmc['y81']==1),'rprice'].mean()
m_t0 = kielmc.loc[(kielmc['nearinc']==1)&(kielmc['y81']==0),'rprice'].mean()
m_c1 = kielmc.loc[(kielmc['nearinc']==0)&(kielmc['y81']==1),'rprice'].mean()
m_c0 = kielmc.loc[(kielmc['nearinc']==0)&(kielmc['y81']==0),'rprice'].mean()
DiD_manual = (m_t1 - m_t0) - (m_c1 - m_c0)
print(f'Treated (near incinerator): 1978=${m_t0:,.0f}  1981=${m_t1:,.0f}  Δ=${m_t1-m_t0:+,.0f}')
print(f'Control (not near):         1978=${m_c0:,.0f}  1981=${m_c1:,.0f}  Δ=${m_c1-m_c0:+,.0f}')
print(f'DiD (ATT estimate):         ${DiD_manual:+,.0f}')

### Step 4.3 — DiD via OLS

In [ ]:
n_k = len(kielmc)
X_did = np.column_stack([np.ones(n_k), kielmc['nearinc'].values,
                         kielmc['y81'].values, kielmc['nearinc_y81'].values])
y_did = kielmc['rprice'].values
beta_did = np.linalg.solve(X_did.T@X_did, X_did.T@y_did)
print(f'DiD OLS delta (manual): ${beta_did[3]:+,.2f}')

In [ ]:
res_did = smf.ols('rprice ~ nearinc + y81 + nearinc_y81', data=kielmc).fit()
delta_sm = res_did.params['nearinc_y81']
print(f'DiD OLS delta (sm):     ${delta_sm:+,.2f}')
print(f'Match manual vs OLS:    {np.isclose(beta_did[3], delta_sm)}')
print(f'Match DiD groups vs OLS:{np.isclose(DiD_manual, delta_sm)}')

### Step 4.4 — Parallel trends: visual check

In [ ]:
means = kielmc.groupby(['nearinc','y81'])['rprice'].mean().unstack()
fig, ax = plt.subplots(figsize=(7,5))
ax.plot([1978,1981], [means.loc[0,0], means.loc[0,1]], 'bs-', label='Control (not near)', lw=2)
ax.plot([1978,1981], [means.loc[1,0], means.loc[1,1]], 'ro-', label='Treated (near)', lw=2)
cf = means.loc[1,0] + (means.loc[0,1] - means.loc[0,0])
ax.plot([1978,1981],[means.loc[1,0], cf], 'r--', alpha=0.5, label='Treated counterfactual')
ax.annotate('', xy=(1981, means.loc[1,1]), xytext=(1981, cf),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1981.1, (means.loc[1,1]+cf)/2, f'ATT=${DiD_manual:,.0f}', color='green', va='center')
ax.set_xticks([1978,1981])
ax.set_ylabel('Mean House Price ($)')
ax.set_title('DiD: Incinerator Effect on House Prices')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch21_02_did.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5 — Regression Discontinuity Design (RDD)

### 5.1 Idea

When treatment is assigned based on a **cutoff** in a running variable $X$:

$$D_i = \mathbf{1}[X_i \geq c]$$

Individuals just below and just above the cutoff $c$ are **similar** (quasi-random),
so comparing them estimates the **Local Average Treatment Effect** at the cutoff.

**Sharp RDD** estimand:

$$\tau_{RDD} = \lim_{x \downarrow c} E[Y_i \mid X_i = x] - \lim_{x \uparrow c} E[Y_i \mid X_i = x]$$

### 5.2 Estimation

Run OLS separately on each side of the cutoff (or with interaction):

$$Y_i = \alpha + \tau D_i + \beta (X_i - c) + \gamma D_i (X_i - c) + \varepsilon_i$$

$\tau$ = jump at the cutoff = RDD treatment effect.

### 5.3 Validity: No sorting around the cutoff

The key assumption is that agents **cannot precisely manipulate** their running variable
to be just above (or just below) the cutoff. Test: density of $X$ should be smooth at $c$.

### Step 5.1 — Simulate sharp RDD (scholarship based on test score)

In [ ]:
np.random.seed(55)
n_rdd = 2000
cutoff = 50
true_tau_rdd = 10.0  # scholarship effect on earnings
score = np.random.uniform(20, 80, n_rdd)
D_rdd = (score >= cutoff).astype(int)
earnings = 30 + 0.5*score + true_tau_rdd*D_rdd + np.random.normal(0, 5, n_rdd)
df_rdd = pd.DataFrame({'earnings': earnings, 'score': score, 'D': D_rdd,
                       'score_c': score - cutoff})
print(f'Cutoff: {cutoff}, True RDD effect: {true_tau_rdd}')
print(f'Treated (above cutoff): {D_rdd.sum()}, Untreated: {(1-D_rdd).sum()}')

### Step 5.2 — Visualise the discontinuity

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
bins = np.arange(20, 82, 3)
for side, color, label in [(score<cutoff,'blue','Below cutoff'), (score>=cutoff,'red','Above cutoff')]:
    b_means = [earnings[(score >= b) & (score < b+3) & side].mean()
               for b in bins if len(earnings[(score >= b) & (score < b+3) & side]) > 0]
    b_centers = [b+1.5 for b in bins if len(earnings[(score >= b) & (score < b+3) & side]) > 0]
    ax.scatter(b_centers, b_means, color=color, s=50, label=label)
ax.axvline(cutoff, color='black', linestyle='--', linewidth=2, label=f'Cutoff={cutoff}')
ax.set_xlabel('Test Score (running variable)')
ax.set_ylabel('Mean Earnings')
ax.set_title('RDD: Discontinuity in Earnings at Scholarship Cutoff')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch21_03_rdd.png'), dpi=150, bbox_inches='tight')
plt.show()

### Step 5.3 — Estimate RDD effect manually (OLS with interaction)

In [ ]:
n_rdd_n = len(df_rdd)
X_rdd = np.column_stack([
    np.ones(n_rdd_n),
    df_rdd['D'].values,
    df_rdd['score_c'].values,
    df_rdd['D'].values * df_rdd['score_c'].values
])
y_rdd = df_rdd['earnings'].values
beta_rdd = np.linalg.solve(X_rdd.T@X_rdd, X_rdd.T@y_rdd)
print(f'RDD effect (manual OLS): {beta_rdd[1]:.3f}')
print(f'True effect:             {true_tau_rdd}')

In [ ]:
res_rdd = smf.ols('earnings ~ D + score_c + D:score_c', data=df_rdd).fit()
print(f'RDD effect (statsmodels): {res_rdd.params["D"]:.3f}')
print(f'Match: {np.isclose(beta_rdd[1], res_rdd.params["D"])}')
print(res_rdd.summary())

### Step 5.4 — Bandwidth sensitivity (local linear, window around cutoff)

In [ ]:
print('RDD estimate by bandwidth (local linear regression):')
for bw in [5, 10, 15, 20, 30]:
    mask = np.abs(df_rdd['score_c']) <= bw
    df_bw = df_rdd[mask]
    if len(df_bw) < 50:
        continue
    r = smf.ols('earnings ~ D + score_c + D:score_c', data=df_bw).fit()
    print(f'  BW={bw:3d}: tau={r.params["D"]:6.3f}  SE={r.bse["D"]:.3f}  n={len(df_bw)}')

---
## 6 — Propensity Score Matching (PSM)

### 6.1 Idea

Under **conditional independence** (selection on observables):

$$Y_i(0), Y_i(1) \perp D_i \mid X_i$$

Conditioning on all covariates $X_i$ eliminates selection bias.

**Propensity score:** $p(X_i) = P(D_i = 1 \mid X_i)$

**Rosenbaum-Rubin theorem:** if $Y_i(d) \perp D_i \mid X_i$, then also $Y_i(d) \perp D_i \mid p(X_i)$.

So we only need to match on the **scalar** $p(X_i)$ rather than all of $X_i$.

### 6.2 Steps

1. Estimate propensity score $\hat{p}(X_i)$ via logit
2. For each treated unit, find the nearest untreated unit in $\hat{p}$
3. Average $Y_i(1) - Y_i(0)$ over matched pairs → ATT estimate

### Step 6.1 — Back to simulated selection data

In [ ]:
# Use self-selected treatment from Section 2
df_ps = df_po[['Y_obs','D','ability']].copy()
print('Selection data (first 5 rows):')
print(df_ps.head())

### Step 6.2 — Estimate propensity score via logit

In [ ]:
res_pscore = smf.logit('D ~ ability', data=df_ps).fit(disp=False)
df_ps['pscore'] = res_pscore.predict()
print('Propensity score (first 5):')
print(df_ps[['D','ability','pscore']].head())
print(f'P-score range: [{df_ps.pscore.min():.3f}, {df_ps.pscore.max():.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(df_ps.loc[df_ps['D']==1,'pscore'], bins=30, alpha=0.6, label='Treated', density=True)
ax.hist(df_ps.loc[df_ps['D']==0,'pscore'], bins=30, alpha=0.6, label='Untreated', density=True)
ax.set_xlabel('Estimated Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Common Support Check: Overlap in Propensity Scores')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch21_04_pscore.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Common support: both distributions should overlap')

### Step 6.3 — Nearest-neighbour matching manually

In [ ]:
treated   = df_ps[df_ps['D']==1].reset_index(drop=True)
untreated = df_ps[df_ps['D']==0].reset_index(drop=True)
matched_diffs = []
for i in range(len(treated)):
    ps_i = treated.loc[i,'pscore']
    # Find nearest untreated by propensity score
    dist = np.abs(untreated['pscore'] - ps_i)
    j    = dist.idxmin()
    diff = treated.loc[i,'Y_obs'] - untreated.loc[j,'Y_obs']
    matched_diffs.append(diff)
ATT_psm = np.mean(matched_diffs)
print(f'ATT (PSM, nearest-neighbour): {ATT_psm:.3f}')
print(f'True ATT:                     {ATT_true:.3f}')
print(f'Naive difference:             {naive_diff:.3f}  (biased by {selection_bias:.3f})')

---
## 7 — Local Average Treatment Effect (LATE) from IV

### 7.1 What IV estimates

When treatment is endogenous, IV with instrument $Z$ identifies the **LATE**:

$$LATE = E[Y_i(1) - Y_i(0) \mid \text{complier}]$$

**Compliers** are individuals whose treatment status is **changed by the instrument**:
$D_i(Z=1) = 1$ and $D_i(Z=0) = 0$.

**IV does NOT estimate ATE** unless treatment effects are homogeneous or everyone is a complier.

| Group | $D(Z=0)$ | $D(Z=1)$ | Included in LATE? |
|---|---|---|---|
| Always-takers | 1 | 1 | No |
| Never-takers | 0 | 0 | No |
| **Compliers** | 0 | 1 | **Yes** |
| Defiers | 1 | 0 | No (excluded by monotonicity) |

### 7.2 The Wald Estimator (simple IV)

With binary $D$ and binary $Z$:

$$LATE = \hat{\beta}_{IV} = \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[D \mid Z=1] - E[D \mid Z=0]} = \frac{\text{Reduced form}}{\text{First stage}}$$

### Step 7.1 — Simulate LATE with binary instrument

In [ ]:
np.random.seed(77)
n_late = 2000
ability_l = np.random.normal(0, 1, n_late)
Z = np.random.binomial(1, 0.5, n_late)   # binary instrument (e.g. randomised offer)
# Heterogeneous treatment effects: LATE only for compliers
true_LATE = 4.0   # compliers' treatment effect
true_ATE  = 2.5   # all individuals average
# Compliance: prob of taking treatment increases with Z
prob_D_z0 = 0.2 + 0.3*(ability_l > 0)  # always-takers + some compliers without offer
prob_D_z1 = 0.5 + 0.3*(ability_l > 0)  # always-takers + compliers WITH offer
D_l = np.where(Z==1, np.random.binomial(1, np.clip(prob_D_z1,0,1)),
                     np.random.binomial(1, np.clip(prob_D_z0,0,1)))
u_l = np.random.normal(0, 2, n_late)
Y_l = 10 + 2*ability_l + D_l*true_LATE + u_l
df_late = pd.DataFrame({'Y': Y_l, 'D': D_l, 'Z': Z, 'ability': ability_l})
print('Share treated by Z:')
print(df_late.groupby('Z')['D'].mean())

### Step 7.2 — Wald estimator manually

In [ ]:
EY_Z1 = df_late.loc[df_late['Z']==1,'Y'].mean()
EY_Z0 = df_late.loc[df_late['Z']==0,'Y'].mean()
ED_Z1 = df_late.loc[df_late['Z']==1,'D'].mean()
ED_Z0 = df_late.loc[df_late['Z']==0,'D'].mean()
reduced_form = EY_Z1 - EY_Z0
first_stage  = ED_Z1 - ED_Z0
LATE_wald = reduced_form / first_stage
print(f'Reduced form (E[Y|Z=1]-E[Y|Z=0]): {reduced_form:.3f}')
print(f'First stage  (E[D|Z=1]-E[D|Z=0]): {first_stage:.3f}')
print(f'Wald LATE = {reduced_form:.3f} / {first_stage:.3f} = {LATE_wald:.3f}')
print(f'True LATE: {true_LATE}')

### Step 7.3 — 2SLS = Wald estimator (verify)

In [ ]:
n_l = len(df_late)
# Stage 1: D ~ Z
res_s1_late = smf.ols('D ~ Z', data=df_late).fit()
D_hat = res_s1_late.fittedvalues.values
# Stage 2: Y ~ D_hat
X_s2_late = np.column_stack([np.ones(n_l), D_hat])
beta_2sls_late = np.linalg.solve(X_s2_late.T@X_s2_late, X_s2_late.T@Y_l)
print(f'2SLS LATE (manual): {beta_2sls_late[1]:.3f}')
print(f'Wald LATE:          {LATE_wald:.3f}')
print(f'Match: {np.isclose(beta_2sls_late[1], LATE_wald)}')

---
## 8 — Comparing All Methods: What Do They Estimate?

| Method | Estimand | Key Assumption | When valid |
|---|---|---|---|
| Naive OLS | Biased (ATE + selection bias) | None needed | Only if no selection |
| RCT / OLS | ATE (= ATT = ATU) | Random assignment | Randomised experiments |
| DiD | ATT | Parallel trends | Panel with pre/post data |
| RDD | LATE at cutoff | No sorting at cutoff | Cutoff-based assignment |
| PSM | ATT | Conditional independence | Selection on observables only |
| IV / 2SLS | LATE (compliers only) | Instrument validity | Valid instrument available |

In [ ]:
# Collect all estimates from this notebook
summary = pd.DataFrame({
    'Method': ['Naive (biased)','RCT','DiD','RDD','PSM','Wald IV'],
    'Estimate': [
        naive_diff,
        res_rct.params['D'],
        res_did.params['nearinc_y81'] / 1000,   # scale for readability
        res_rdd.params['D'],
        ATT_psm,
        LATE_wald
    ],
    'Estimand': ['ATE + bias','ATE','ATT','LATE @ cutoff','ATT','LATE (compliers)']
})
print(summary.round(3).to_string(index=False))
summary.to_csv(os.path.join(output_dir,'table_ch21_te_comparison.csv'), index=False)
print('\nSaved table_ch21_te_comparison.csv')

---
## 9 — Key Takeaways

1. **The fundamental problem:** We never observe both $Y(1)$ and $Y(0)$ for the same person.
All causal methods are strategies to construct valid **counterfactuals**.

2. **ATE ≠ ATT ≠ LATE** — different methods answer different questions about different populations.
Always be clear which estimand your method targets.

3. **Selection bias distorts OLS** when high-ability people self-select into treatment.
The naive estimate inflates the treatment effect.

4. **RCT is the gold standard** — random assignment eliminates selection bias by design.
Adding controls only improves precision, not the point estimate.

5. **DiD removes time-invariant selection** under parallel trends.
The counterfactual for the treated group is inferred from the control group's trend.

6. **RDD is very credible locally** — individuals near the cutoff are nearly identical.
But LATE at the cutoff may not generalise far from it (external validity concern).

7. **PSM requires selection on observables** — if unobserved confounders exist, PSM fails.
It is only as good as the set of covariates you can match on.

8. **IV/2SLS estimates LATE for compliers only** — not the same as ATE unless effects are homogeneous.
Always report which subpopulation is relevant for your research question.